# Executive Summary

**Objective:** 
To integrate geographical metadata, resolve string formatting inconsistencies, and engineer normalized metrics, transforming the raw data into a finalized state ready for exploratory data analysis.

**Data Flow:**
*   **Inputs:** 
    * `data\raw\internship_positions.parquet`
    * `data\external\administrative_divisions.parquet`
*   **Output:** `data\interim\internship_postings.parquet` (Exported to the interim directory to preserve datatypes)

**Key Operations Performed:**
1. **Data Integration (Province Mapping) [<u>[click]</u>](#1-data-integration):** Mapped raw job locations to their respective provinces by joining against the administrative divisions dictionary.
    * **String Cleaning [<u>[click]</u>](#11-string-cleaning):** Resolved mismatching geographic keys by standardizing text (lowercasing, stripping punctuation, and removing extraneous words).
    * **Manual Overrides [<u>[click]</u>](#12-manual-overrides):** Applied a manual dictionary mapping to achieve 100% province mapping with zero null values.
2. **Data Conversion [<u>[click]</u>](#2-data-conversion):** Cast `weekly_working_day` to a categorical datatype due to its low cardinality (2 unique values).
3. **Feature Engineering [<u>[click]</u>](#3-feature-engineering):**
    * Engineered the `acceptance_percentage` column (`100 * approved_quota / (1 + applicant_count)`).
    * Categorized `job_title` into a new `job_category` feature to reduce cardinality.
    * Engineered 9 new boolean flag columns (e.g., `allows_it_and_computer_majors`) by grouping over 1,300 distinct majors in the `allowed_major` column using regex pattern matching.
    * Binned all heavy right-skewed numericals (`requested_quota`, `approved_quota`, `applicant_count`, `acceptance_percentage`) to capture "whale" postings in an "Extreme" category without deleting them.
    * One-hot encoded `education_level` for downstream stakeholder consumption.
    * Extracted a new binary flag, `allows_all_majors`, by parsing the `job_description` column.
4. **Schema Finalization [<u>[click]</u>](#4-schema-finalization):** Reorganized the final 14 columns into a logical analytical structure before exporting.

# Setup & Imports

In [49]:
# Import libraries
import numpy as np
import pandas as pd

from src.config import EXTERNAL_DATA_DIR, INTERIM_DATA_DIR, RAW_DATA_DIR

In [50]:
# Load datasets
adm_divisions = pd.read_parquet(EXTERNAL_DATA_DIR / "administrative_divisions.parquet")
internship_positions = pd.read_parquet(RAW_DATA_DIR / "internship_positions.parquet")

# 1. Data Integration
Mapping raw job locations to their respective provinces by joining against the administrative divisions dictionary.

In [51]:
# Convert a regency-province table into a dictionary
adm_divisions_dict = dict(zip(adm_divisions["regency"], adm_divisions["province"]))


In [52]:
# Create a new column, `province`, by mapping with a dictionary
internship_positions["province"] = internship_positions["job_location"].map(adm_divisions_dict)
display(internship_positions.head())

,job_id,published_at,job_title,company,job_location,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,1,0,Sumatera Utara
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Sumatera Utara
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,1,0,Jawa Tengah
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,1,0,Kalimantan Tengah
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Jawa Timur


In [53]:
# Find regencies from `internship_positions` that have no match 
# with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

job_location
Kota Batam                          221
Kota Palangkaraya                   120
Kota Dumai                           77
Kab. Pangkajene Kepulauan            57
Kota Bau Bau                         56
Kota Banjarbaru                      48
Kab. Siak                            46
Kota Lubuk Linggau                   43
Kab. Gunungkidul                     42
Kab. Karangasem                      39
Kota Sawahlunto                      38
Kab. Batanghari                      31
Kab. Tulang Bawang                   28
Kota Pematangsiantar                 25
Unknown Location                     24
Kab. Kotabaru                        24
Kab. Banyuasin                       23
Kab. Labuhanbatu                     23
Kota Pare Pare                       20
Kab. Tojo Una Una                    18
Kab. Pahuwato                        17
Kab. Kep. Siau Tagulandang Biaro     17
Kepulauan Tanimbar                   15
Kab Timor Tengah Selatan             15
Kab. Toli Toli             

In [54]:
# Get all regencies from `adm_divisions` that have no match
# with job locations from `internship_positions`
matched_regencies = list(set(internship_positions[
    ~internship_positions.province.isnull()
]["job_location"].to_list()))

unmatched_regencies = list(set(adm_divisions.regency.to_list()) - set(matched_regencies))

display(
    adm_divisions[
        adm_divisions.regency.isin(unmatched_regencies)
    ][["regency", "province"]].sort_values("regency", ascending=False)
)

,regency,province
70,Kota Sawah Lunto,Sumatera Barat
50,Kota Pematang Siantar,Sumatera Utara
420,Kota Parepare,Sulawesi Selatan
341,Kota Palangka Raya,Kalimantan Tengah
114,Kota Lubuklinggau,Sumatera Selatan
...,...,...
90,Kab. Batang Hari,Jambi
104,Kab. Banyu Asin,Sumatera Selatan
144,Kab. Bangka Selatan,Kepulauan Bangka Belitung
385,Kab. Banggai Kepulauan,Sulawesi Tengah


## 1.1 String Cleaning

In [55]:
# Fallback mapping: Stripping punctuation and whitespace resolves mismatches
# caused by inconsistent data entry on the scraped platform.
plain_adm_div_dict = dict(
    zip(
        adm_divisions["regency"]
        .str.lower()
        .str.replace(r"\sdan\s", " ", case=False, regex=True)
        .str.replace(r"\W", "", regex=True),
        adm_divisions["province"],
    )
)

display(plain_adm_div_dict)

mask = internship_positions["province"].isnull()
internship_positions.loc[mask, "province"] = (
    internship_positions.loc[mask, "job_location"]
    .str.lower()
    .str.replace(r"\sdan\s", " ", case=False, regex=True)
    .str.replace(r"\W", "", regex=True)
    .map(plain_adm_div_dict)
)

{'kabsimeulue': 'Aceh',
 'kabacehsingkil': 'Aceh',
 'kabacehselatan': 'Aceh',
 'kabacehtenggara': 'Aceh',
 'kabacehtimur': 'Aceh',
 'kabacehtengah': 'Aceh',
 'kabacehbarat': 'Aceh',
 'kabacehbesar': 'Aceh',
 'kabpidie': 'Aceh',
 'kabbireuen': 'Aceh',
 'kabacehutara': 'Aceh',
 'kabacehbaratdaya': 'Aceh',
 'kabgayolues': 'Aceh',
 'kabacehtamiang': 'Aceh',
 'kabnaganraya': 'Aceh',
 'kabacehjaya': 'Aceh',
 'kabbenermeriah': 'Aceh',
 'kabpidiejaya': 'Aceh',
 'kotabandaaceh': 'Aceh',
 'kotasabang': 'Aceh',
 'kotalangsa': 'Aceh',
 'kotalhokseumawe': 'Aceh',
 'kotasubulussalam': 'Aceh',
 'kabnias': 'Sumatera Utara',
 'kabmandailingnatal': 'Sumatera Utara',
 'kabtapanuliselatan': 'Sumatera Utara',
 'kabtapanulitengah': 'Sumatera Utara',
 'kabtapanuliutara': 'Sumatera Utara',
 'kabtobasamosir': 'Sumatera Utara',
 'kablabuhanbatu': 'Sumatera Utara',
 'kabasahan': 'Sumatera Utara',
 'kabsimalungun': 'Sumatera Utara',
 'kabdairi': 'Sumatera Utara',
 'kabkaro': 'Sumatera Utara',
 'kabdeliserdang': '

In [56]:
# Check for the missing values again by finding regencies from `internship_positions`
# that have no match with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

job_location
Unknown Location                    24
Kab. Kep. Siau Tagulandang Biaro    17
Kab. Pahuwato                       17
Kepulauan Tanimbar                  15
Kab. Mahakam Ulu                     1
Name: job_id, dtype: int64

## 1.2 Manual Overrides

In [57]:
# Manual overrides for edge cases missing from the standard division dataset
manual_dict = {
    "Kab. Kep. Siau Tagulandang Biaro": "Sulawesi Utara",
    "Kab. Mahakam Ulu": "Kalimantan Timur",
    "Kab. Pahuwato": "Gorontalo",
    "Kepulauan Tanimbar": "Maluku",
    "Unknown Location": "Unknown Location",
}

mask = internship_positions["province"].isnull()
internship_positions.loc[mask, "province"] = internship_positions.loc[
    mask, "job_location"
].map(manual_dict)

In [58]:
# Check for the missing values again by finding regencies from `internship_positions`
# that have no match with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

Series([], Name: job_id, dtype: int64)

In [59]:
# Rename Column `job_location` to `regency_city`
internship_positions.rename(columns={"job_location": "regency_city"}, inplace=True)

In [60]:
# Fix some regency and city names
internship_positions["regency_city"] = (
    internship_positions.regency_city
    .str.replace(r"^Kab\s", r"Kab. ", regex=True)
    .str.replace("Pahuwato", "Pohuwato")
    .str.replace(r"^Kepulauan\s", r"Kab. Kep. ", regex=True)
    .str.replace(r"\sKepulauan\s", r" Kep. ", regex=True)
)

# 2. Data Conversion
Casting `weekly_working_day` to a categorical datatype due to its low cardinality (2 unique values).

In [61]:
# Cast the data type of `weekly_working_day` to string
internship_positions = internship_positions.astype({"weekly_working_day": "str"})

internship_positions.info()

<class 'pandas.DataFrame'>
RangeIndex: 28322 entries, 0 to 28321
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   job_id              28322 non-null  str  
 1   published_at        28322 non-null  str  
 2   job_title           28322 non-null  str  
 3   company             28322 non-null  str  
 4   regency_city        28322 non-null  str  
 5   education_level     28322 non-null  str  
 6   allowed_major       28322 non-null  str  
 7   job_description     28322 non-null  str  
 8   weekly_working_day  28322 non-null  str  
 9   requested_quota     28322 non-null  int64
 10  approved_quota      28322 non-null  int64
 11  applicant_count     28322 non-null  int64
 12  province            28322 non-null  str  
dtypes: int64(3), str(10)
memory usage: 20.0 MB


# 3. Feature Engineering

## 3.1 Feature Construction
Constructing the `acceptance_percentage` column (`100 * approved_quota / (1 + applicant_count)`)

In [62]:
# Create Column `acceptance_percentage`
internship_positions["acceptance_percentage"] = round(
    100
    * internship_positions["approved_quota"]
    / (internship_positions["applicant_count"].add(1)),
    2,
)

display(internship_positions.sample(10))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province,acceptance_percentage
18876,a242b047-bcd3-4cc2-98a5-74de13ef8873,2026-07-16T12:39:43+07:00,Desainer Multimedia,BBPVP Serang,Kota Serang,"Diploma, Bachelor","Multimedia, Ilmu Komputer, Animasi, Desain Kom...",1. Membantu pembuatan desain grafis untuk kebu...,5,1,1,10,Banten,9.09
18682,a2438be7-cfb1-452d-aeaa-303cf565d899,2026-07-16T10:54:44+07:00,BNI Digital Assisstan (BDA),Perusahaan Perseroan (Persero) Bank Negara Ind...,Kab. Purbalingga,"Diploma, Bachelor","Perbankan Dan Keuangan, Komunikasi, Manajemen,...",melayani dan mengedukasi nasabah terkait trans...,5,4,4,39,Jawa Tengah,10.00
1438,a2434755-2151-407b-a6e9-f4b12fc89621,2026-07-16T12:43:14+07:00,Okupasi Terapi,Rumah Sakit Umum Pusat Dr. M. Djamil Padang,Kota Padang,"Diploma, Bachelor","Okupasi Terapi, Terapi Okupasi",1. Memberikan layanan okupasi terapi pada pasi...,5,2,2,5,Sumatera Barat,33.33
1790,a242c659-6a6d-4e82-9c4f-4917c61a5a2b,2026-07-16T13:00:34+07:00,ASISTEN PEMBINAAN KEPRIBADIAN TARI,RUMAH TAHANAN NEGARA KELAS IIB MENGGALA,Kab. Tulang Bawang,Bachelor,"Seni Tari, Pendidikan Seni Tari",1.\tMenyusun dan melaksanakan program pembinaa...,6,1,1,3,Lampung,25.00
27141,a242e3a5-60bb-4f7d-950c-4bd3ee2f781a,2026-07-16T12:37:38+07:00,Asisten Analis SDM dan Hukum,BPS Provinsi RIAU,Kota Pekanbaru,"Bachelor, Diploma","Manajemen Sumber Daya Manusia, Manajemen, Psik...","Mendukung kegiatan administrasi kepegawaian, p...",5,3,3,69,Riau,4.29
27059,a244099e-07b0-4b28-91ce-a28ebedfe129,2026-07-16T11:55:58+07:00,DKP – Asisten Analisis Kebijakan Kelautan & Pe...,Deputi Bidang Kebijakan Pembangunan,Kota Adm. Jakarta Pusat,"Diploma, Bachelor","Teknik Lingkungan, Ilmu Kelautan, Perikanan, P...","Merencanakan perumusan kebijakan kemaritiman, ...",5,1,1,23,DKI Jakarta,4.17
954,a243d1fb-d58d-4cfc-bead-b9fdef3d1ee2,2026-07-16T12:00:58+07:00,PERAWAT,LEMBAGA PEMASYARAKATAN KELAS IIB KOTA BAKTI,Kab. Pidie,Bachelor,Keperawatan,"""1. Memberikan perawatan medis dasar bagi pega...",6,1,1,2,Aceh,33.33
24670,a2412b52-d179-49ab-8bc8-979fd3befacf,2026-07-16T12:35:22+07:00,Asisten Arsiparis,BPS Kabupaten Madiun,Kab. Madiun,"Diploma, Bachelor","Kearsipan, Kearsipan Digital, Informasi, Perpu...","Mendukung penyusunan, pengelolaan, penyimpanan...",5,1,1,16,Jawa Timur,5.88
14500,a243df61-c2f8-465d-9ece-7953345dc862,2026-07-16T12:03:45+07:00,Analis Sumber Daya Manusia pada Biro SDM dan O...,Kementerian Pemberdayaan Perempuan dan Perlind...,Kota Adm. Jakarta Pusat,Bachelor,"Manajemen Sumber Daya Manusia, Psikologi, Ilmu...","Melakukan pengumpulan, pengolahan, dan analisi...",5,3,3,21,DKI Jakarta,13.64
25762,a243a580-4b3c-4e08-9ee6-af8a73f0e8ee,2026-07-16T10:54:04+07:00,Staff Perencanaan Peralatan Pelabuhan - Kantor...,PT Pelindo Terminal Petikemas,Kota Surabaya,"Diploma, Bachelor","Teknik Mekatronika, Fisika, Teknik Mesin, Tekn...",1. Mendukung penyusunan perencanaan kebutuhan ...,5,1,1,18,Jawa Timur,5.26


## 3.2 Feature Transformation
### 3.2.1 Text Classification
* Categorizing `job_title` into a new `job_category` feature to reduce cardinality.

In [63]:
# Create a custom function to categorize the jobs
def categorize_job(title):
    if pd.isna(title):
        return "Uncategorized"
    
    t = str(title).lower()
    
    # 1. Healthcare & Medical (Added severe typos, hospital codes, and specialized terms)
    if any(w in t for w in ["perawat", "ners", "nurse", "psikiat", "spikiat", "psikat", "pskiat", "psikolog", "piskolog", "pskolog", "medis", "medic", "medik", "gizi", "nutri", "diet", "dokter", "doker", "doktor", "bidan", "apotek", "aptoker", "farmasi", "pharmac", "fisio", "physio", "radio", "sanitari", "sanitasi", "kesehatan", "promkes", "okupasi", "elektromedis", "atem", "terapi", "therap", "klinik", "atlm", "epidemiolog", "anestesi", "anastesi", "rekam medi", "perekam", "mr ", "casemix", "coder", "koder", "cssd", "ipsrs", "audiolog", "orthotic", "mcu", "ranap", "igd", "poliklinik", "vk ", "bersalin", "hemodialisa", "kardiovaskuler", "cardiovascular", "refraksi", "kebidanan", "keperawatan", "patologi", "mikrobiologi", "imunologi", "darah", "ambul", "hospital", "rehabilitasi", "admission"]):
        return "Healthcare & Medical"
    
    # 2. IT & Data
    elif any(w in t for w in ["komputer", "it ", " it", "programmer", "developer", "software", "data", "sistem", "system", "ui/", "/ux", "network", "cyber", "aplikasi", "application", "backend", "frontend", "website", "web", "informatika", "pusdatin", "jaringan", "ai ", "machine learning", "bda", "digital", "erp", "sap ", "helpdesk", "support it", "noc ", "rpa ", "command center", "dashboard", "cloud", "iot", "analytic"]):
        return "IT & Data"
        
    # 3. Engineering & Maintenance
    elif any(w in t for w in ["teknis", "technician", "maintenance", "engineer", "mekanik", "mechanic", "drafter", "drawing", "listrik", "sipil", "civil", "bangunan", "hvac", "otomotif", "mesin", "machine", "welder", "welding", "proyek", "project", "elektro", "electric", "arsitek", "architect", "maint", "equipment", "facility", "sarana", "prasarana", "geologi", "tambang", "mining", "instrument", "surveyor", "craft", "plumbing", "geofisika", "seismik", "geodesi", "geomatika", "toolman", "inspector", "inspektur", "konstruksi", "construction", "pipa", "baja", "otomasi", "automation"]):
        return "Engineering & Maintenance"
        
    # 4. Manufacturing, QA & Production
    elif any(w in t for w in ["produksi", "production", "operator", "qc", "qa", "quality", "pabrik", "manufacturing", "assembly", "packaging", "mutu", "plant", "molding", "mould", "mold", "pattern maker", "slitting", "blown film", "sewing", "garment", "textile", "printing", "finishing", "curing", "mixing", "extruder", "ppic", "rnd", "r&d", "research", "reserch", "set up", "cleanning", "rewinding", "laminasi", "improvement", "pdca", "koe ", "lean", "mill", "helper", "pe ", "ie "]):
        return "Manufacturing, QA & Production"
        
    # 5. Finance & Banking
    elif any(w in t for w in ["keuangan", "akuntansi", "accounting", "akuntan", "pajak", "tax", "auditor", "audit", "bendahara", "anggaran", "finance", "treasury", "billing", "kasir", "cashier", "credit", "kredit", "loan", "pembiayaan", "funding", "transaction", "collection", "receivable", "payable", "wealth", "insurance", "asuransi", "actuary", "aktuaria", "bank", "bni", "teller", "pawning", "micro", "invest", "budget", "cost "]):
        return "Finance & Banking"
        
    # 6. Sales, Marketing & Hospitality
    elif any(w in t for w in ["barista", "cook", "koki", "pastry", "bakery", "culinary", "chef", "layanan", "frontliner", "frontlner", "sales", "marketing", "f&b", "fb ", "store", "customer", "receptionist", "pemasaran", "pramusaji", "hotel", "event", "reservation", "guest", "hospitality", "dancer", "entertainment", "commercial", "merchandis", "retail", "promot", "promosi", "brand", "business development", "bd ", "partnership", "account executive", "masak", "food", "beverage", "catering", "kitchen", "tour ", "travel", "room", "bro", "activation"]):
        return "Sales, Marketing & Hospitality"
        
    # 7. Media, PR & Creative
    elif any(w in t for w in ["humas", "kehumasan", "design", "desain", "kreatif", "creative", "video", "animator", "animation", "content", "konten", "sosial media", "social media", "sosmed", "kol ", "publisitas", "jurnalis", "journalist", "wartawan", "editor", "multimedia", "reporter", "fotografer", "photographer", "broadcasting", "komunikasi", "communication", "visual", "vm artist", "motion", "copywriter", "writer", "art ", "talent", "audio", "camera", "campaign", "publikasi", "publik", "illustrator", "media", "broadcast", "creator"]):
        return "Media, PR & Creative"
        
    # 8. Legal, Risk & Compliance
    elif any(w in t for w in ["hukum", "legal", "law", "compliance", "kepatuhan", "risk", "risiko", "hse", "hsse", "qhse", "she", "ehs", "safety", "k3", "security", "keamanan", "fraud", "investigasi", "pengaduan", "maladministrasi", "litigasi", "regulas", "regulatory", "sertifikasi", "perizinan", "izin", "kekayaan intelektual"]):
        return "Legal, Risk & Compliance"
        
    # 9. Logistics & Supply Chain
    elif any(w in t for w in ["gudang", "warehouse", "logistik", "logistic", "exim", "supply chain", "scm", "inventory", "pengadaan", "purchasing", "procurement", "buyer", "ekspor", "impor", "export", "import", "cargo", "shipping", "freight", "delivery", "transport", "fleet", "ekspeditor", "harbour", "port ", "bandara", "airport", "pelabuhan", "aviation", "aero", "aircraft", "checker", "terminal"]):
        return "Logistics & Supply Chain"
        
    # 10. Education, Training & Government
    elif any(w in t for w in ["kebijakan", "pemerintahan", "penelaah", "pengawas", "asn", "biro", "kementerian", "pemda", "pns", "diplomat", "instruktur", "pelatihan", "pembelajaran", "tentor", "pengajar", "edukator", "diklat", "akademik", "guru", "dosen", "widyaiswara", "pusat", "badan", "tutor", "statistik", "statistisi", "peneliti", "pustaka", "kearsipan", "arsip", "kurator", "laporan", "pelaporan", "penyusun", "evaluasi", "pengolah", "dokumen", "evaluator", "pemeriksaan"]):
        return "Education, Training & Government"
        
    # 11. Agriculture & Environment
    elif any(w in t for w in ["pertanian", "perikanan", "peternakan", "perkebunan", "agribisnis", "kehutanan", "lingkungan", "agronomi", "pangan", "tambak", "tanaman", "kebun", "hewan", "hutan", "forestry", "environment", "sustainability", "esg", "limbah", "waste", "marine", "hydro", "iklim", "climate", "budidaya", "ternak", "nelayan", "satwa", "flora", "fauna", "ekologi", "air "]):
        return "Agriculture & Environment"
        
    # 12. Language & Translation
    elif any(w in t for w in ["isyarat", "penerjemah", "translator", "interpreter", "language", "mandarin", "japanese", "english", "bahasa"]):
        return "Language & Translation"
        
    # 13. Correctional & Social Services
    elif any(w in t for w in ["pembinaan", "kepribadian", "pembimbing kemasyarakatan", "warga binaan", "kegiatan kerja", "rohani", "sosial", "pemasyarakatan", "community", "csr", "tjsl", "bina", "klien", "konselor"]):
        return "Correctional & Social Services"

    # 14. HR, Admin & Management (General catch-alls placed at the very end)
    elif any(w in t for w in ["sdm", "human resource", "hr", "ga", "general affair", "administrasi", "admin", "bmn", "sekretaris", "secretary", "tata usaha", "tu ", "personil", "personalia", "rekrutmen", "recruitment", "talent acquisition", "od ", "organization development", "umum", "fasilitas", "manajemen", "management", "manager", "pmo", "strategi", "koordinator", "coordinator", "supervisor", "spv", "director", "operasional", "operation", "asset", "aset", "clerical", "sarana", "pejabat", "pengelola", "asisten", "officer", "staff", "staf", "pelaksana", "magang", "intern", "consultant", "konsultan", "planner"]):
        return "HR, Admin & Management"
        
    else:
        return "Other"

In [64]:
# Apply the function to create the new column
internship_positions["job_category"] = internship_positions["job_title"].apply(categorize_job)

# Verify the distribution of the new categories
display(internship_positions["job_category"].value_counts())

job_category
HR, Admin & Management              5176
Healthcare & Medical                4063
IT & Data                           2802
Media, PR & Creative                2570
Sales, Marketing & Hospitality      2150
Correctional & Social Services      2005
Finance & Banking                   1707
Engineering & Maintenance           1701
Education, Training & Government    1294
Manufacturing, QA & Production      1083
Legal, Risk & Compliance            1036
Other                                994
Logistics & Supply Chain             837
Agriculture & Environment            597
Language & Translation               307
Name: count, dtype: int64

* Engineering 9 new boolean flag columns (e.g., `allows_it_and_computer_majors`) by grouping over 1,300 distinct majors in the `allowed_major` column using regex pattern matching.

In [67]:
# Create new Boolean columns
# Define the categories and their specific Indonesian Regex keywords
maj_categories = {
    "it_and_computer": r"informatika|komputer|sistem informasi|perangkat lunak|multimedia|jaringan|siber|data|teknologi informasi|piranti lunak|website",
    "engineering": r"teknik(?!\s*(?:informatika|komputer|multimedia))|rekayasa(?!\s*(?:perangkat lunak|internet|komputer))|arsitektur|mesin|elektro|sipil|industri|mekatronika|otomotif|manufaktur|konstruksi|geodesi|geologi|tambang|perkapalan|dirgantara|nautika|listrik|kelistrikan|logam|tekstil|metrologi|instrumentasi|perencanaan|planologi|tata ruang",
    "business": r"manajemen|akuntansi|bisnis|ekonomi|keuangan|administrasi|adminsitrasi|logistik|pemasaran|marketing|pajak|perbankan|retail|niaga|aktiva|kewirausahaan|asuransi",
    "health": r"kedokteran|keperawatan|kebidanan|farmasi|kesehatan|gizi|medik|medis|terapi|radiologi|klinik|apoteker|sanitasi|higiene|hiperkes|optisi|optometri|ortotik|prostetik|darah|audiologi|akupunktur|herbal|rumah sakit|nutrisi",
    "science_and_agri": r"matematika|statistik|statistika|biologi|kimia|fisika|sains|agribisnis|agribinis|pertanian|peternakan|perikanan|kehutanan|agroteknologi|agroekoteknologi|agro|perkebunan|agronomi|hortikultura|hewan|laut|oseanografi|aktuaria|geografi|astronomi|lingkungan|budidaya|tanaman|bumi|pangan|pertanahan|kartografi|penginderaan",
    "arts_and_media": r"desain|seni|komunikasi|film|televisi|jurnalistik|penyiaran|broadcasting|hubungan masyarakat|humas|fotografi|kriya|tari|musik|karawitan|animasi|media|audio|video|penerbitan",
    "social_and_law": r"hukum|sosiologi|psikologi|sastra|bahasa|kriminologi|kesejahteraan|pemerintahan|politik|hubungan internasional|sejarah|filsafat|antropologi|perpustakaan|kearsipan|arsip|agama|teologi|syariah|islam|kristen|buddha|hindu",
    "education": r"pendidikan|pgsd|pgpaud|tadris|bimbingan|konseling|tarbiyah|guru|kependidikan|penyuluhan",
    "tourism_and_hospitality": r"pariwisata|perhotelan|tata boga|tata rias|tata busana|fashion|kuliner|wisata|mice|travel|hospitaliti|hidang|patiseri"
}

# Ensure the column is treated as a string and handle missing values
internship_positions["allowed_major"] = internship_positions["allowed_major"].fillna("")

# Iterate through the dictionary to create the new Boolean columns
for cat, pattern in maj_categories.items():
    col_name = f"allows_{cat}_majors"
    
    # Check if any keyword in the pattern exists in the "allowed_major" string
    mask = internship_positions["allowed_major"].str.contains(pattern, case=False, regex=True)    
    internship_positions[col_name] = np.where(mask, "Yes", "No")

# Preview the results
display(internship_positions.sample(10))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,job_category,allows_it_and_computer_majors,allows_engineering_majors,allows_business_majors,allows_health_majors,allows_science_and_agri_majors,allows_arts_and_media_majors,allows_social_and_law_majors,allows_education_majors,allows_tourism_and_hospitality_majors
27756,a243abfd-839e-42cd-8972-96858982ada2,2026-07-16T10:54:05+07:00,Staff HSSE - TTL,PT Pelindo Terminal Petikemas,Kota Surabaya,"Diploma, Bachelor","Teknik Lingkungan, Keselamatan Dan Kesehatan K...","1. Mendukung pelaksanaan implementasi Health, ...",5,1,...,"Legal, Risk & Compliance",No,Yes,No,Yes,Yes,No,No,No,No
12377,a243675c-0b44-409a-88ac-31b2c457b042,2026-07-16T09:56:59+07:00,Administrative Staff Intern - Internal Audit,Pelita Air Service,Kota Adm. Jakarta Pusat,"Diploma, Bachelor","Keuangan, Manajemen, Sistem Informasi, Adminis...",Mendukung pelaksanaan kegiatan Internal Audit ...,5,3,...,Finance & Banking,Yes,No,Yes,No,No,No,No,No,No
24498,a23f805c-765d-4a2d-8b1c-f90965bd9634,2026-07-16T10:34:01+07:00,Airport Non-Aero Commercial Department,PT Angkasa Pura Indonesia Bandar Udara Interna...,Kota Semarang,Bachelor,"Manajemen, Teknik Industri, Ilmu Komunikasi, A...",1. Mendukung pelaksanaan Commercial Operation ...,5,3,...,"Sales, Marketing & Hospitality",No,Yes,Yes,No,No,Yes,No,No,No
28224,a24168fb-37dc-4618-a65d-3ed7352c0c31,2026-07-16T11:58:13+07:00,Administrasi Klaim,BPJS Kesehatan Kantor Cabang Medan,Kota Medan,"Diploma, Bachelor","Keperawatan, Farmasi, Kesehatan Masyarakat, Ad...",Membantu pelaksanaan kegiatan administratif da...,5,1,...,"HR, Admin & Management",No,No,Yes,Yes,No,No,No,No,No
24554,a2410856-3645-4475-a95c-635ce2ffd25f,2026-07-16T10:04:04+07:00,HSE Intern,PT Synergy Oil Nusantara,Kota Batam,"Diploma, Bachelor","Teknik Lingkungan, Kesehatan dan Keselamatan K...",Penerapan Sistem Manajemen HSE Mengimplementas...,6,2,...,"Legal, Risk & Compliance",No,Yes,No,Yes,Yes,No,No,No,No
25429,a232bb64-2252-4226-b869-6b3629888c18,2026-07-16T09:58:08+07:00,EXIM Staff,PT. King Jim Indonesia,Kab. Pasuruan,Bachelor,"Bisnis Logistik, Hubungan Internasional, Manaj...",1. Memahami dan menjalankan sesuai prosedur al...,5,1,...,Logistics & Supply Chain,No,No,Yes,No,No,No,Yes,No,No
17838,a24109a9-1f76-4992-a3c3-c6ed71082c5b,2026-07-16T10:31:45+07:00,Buyer Raw Material Senior Staff,PT. Eka Bogainti,Kota Adm. Jakarta Timur,"Diploma, Bachelor, Profession","Logistik, Manajemen, Teknik Industri, Teknolog...",Bertanggung jawab atas pengadaan bahan baku fr...,5,1,...,Logistics & Supply Chain,No,Yes,Yes,No,Yes,No,No,No,No
999,a241930a-b8c9-4c5b-b2ca-80082f19a7f5,2026-07-16T11:54:31+07:00,JURU BAHASA ISYARAT,KANTOR WILAYAH DIREKTORAT JENDERAL PEMASYARAKA...,Kota Ambon,Bachelor,Pendidikan Luar Biasa,1. Menjadi penerjemah bagi pengguna layanan pe...,5,1,...,Language & Translation,No,No,No,No,No,No,No,Yes,No
11138,a242f057-43f5-41c7-a69d-fcce5dac18a0,2026-07-16T09:53:11+07:00,Automation Engineering Internship,PT. Autoplastik Indonesia,Kab. Karawang,"Diploma, Bachelor","Teknik Otomasi Industri, Mekatronika, Teknik E...",Mendukung transformasi digitalisasi melalui pe...,5,1,...,Engineering & Maintenance,No,Yes,No,No,No,No,No,No,No
3927,a240f193-831e-44be-8d6f-bb0f072b8a3d,2026-07-16T12:01:03+07:00,PERAWAT KESEHATAN,RUMAH TAHANAN NEGARA KELAS IIB DUMAI,Kota Dumai,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,...,Healthcare & Medical,No,No,No,Yes,No,No,No,No,No


In [68]:
# Review the rows that slipped through the Regex patterns
category_cols = [f"allows_{cat}_majors" for cat in maj_categories.keys()]
uncategorized_mask = (internship_positions[category_cols] == "No").all(axis=1)

uncategorized_positions = internship_positions[uncategorized_mask]

print(uncategorized_positions['allowed_major'].unique())

<ArrowStringArray>
[]
Length: 0, dtype: str


### 3.2.2 Binning (Discretization)
Binning all heavy right-skewed numericals (`requested_quota`, `approved_quota`, `applicant_count`, `acceptance_percentage`) to capture "whale" postings in an "Extreme" category without deleting them.

In [69]:
# Bin `requested_quota` and `approved_quota`
quota_edges = [1, 2, 10, 50, np.inf]
quota_labels = ["1 to 2", "3 to 10", "11 to 50", "50+"]

internship_positions["requested_quota_category"] = pd.cut(
    internship_positions["requested_quota"],
    bins=quota_edges,
    labels=quota_labels,
    include_lowest=True
)

internship_positions["approved_quota_category"] = pd.cut(
    internship_positions["approved_quota"],
    bins=quota_edges,
    labels=quota_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_engineering_majors,allows_business_majors,allows_health_majors,allows_science_and_agri_majors,allows_arts_and_media_majors,allows_social_and_law_majors,allows_education_majors,allows_tourism_and_hospitality_majors,requested_quota_category,approved_quota_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,...,No,No,No,No,No,Yes,No,No,1 to 2,1 to 2
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,No,No,Yes,No,No,No,No,No,1 to 2,1 to 2
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,...,No,No,Yes,No,No,No,No,No,1 to 2,1 to 2
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,...,No,No,Yes,No,No,No,No,No,1 to 2,1 to 2
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,No,No,Yes,No,No,No,No,No,1 to 2,1 to 2


In [70]:
# Bin Column `applicant_count`
applicant_edges = [0, 5, 10, 20, 50, np.inf]
applicant_labels = ["0 to 5", "6 to 10", "11 to 20", "21 to 50", "50+"]

internship_positions["applicant_count_category"] = pd.cut(
    internship_positions["applicant_count"],
    bins=applicant_edges,
    labels=applicant_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_business_majors,allows_health_majors,allows_science_and_agri_majors,allows_arts_and_media_majors,allows_social_and_law_majors,allows_education_majors,allows_tourism_and_hospitality_majors,requested_quota_category,approved_quota_category,applicant_count_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,...,No,No,No,No,Yes,No,No,1 to 2,1 to 2,0 to 5
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,No,Yes,No,No,No,No,No,1 to 2,1 to 2,0 to 5
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,...,No,Yes,No,No,No,No,No,1 to 2,1 to 2,0 to 5
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,...,No,Yes,No,No,No,No,No,1 to 2,1 to 2,0 to 5
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,No,Yes,No,No,No,No,No,1 to 2,1 to 2,0 to 5


In [71]:
# Bin Column `acceptance_percentage`
acceptance_edges = [0, 10, 25, 50, np.inf]
acceptance_labels = ["0 - 10%", "11 - 25%", "26 - 50%", "50%+"]

internship_positions["acceptance_percentage_category"] = pd.cut(
    internship_positions["acceptance_percentage"],
    bins=acceptance_edges,
    labels=acceptance_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_health_majors,allows_science_and_agri_majors,allows_arts_and_media_majors,allows_social_and_law_majors,allows_education_majors,allows_tourism_and_hospitality_majors,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,...,No,No,No,Yes,No,No,1 to 2,1 to 2,0 to 5,50%+
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,Yes,No,No,No,No,No,1 to 2,1 to 2,0 to 5,50%+
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,...,Yes,No,No,No,No,No,1 to 2,1 to 2,0 to 5,50%+
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,...,Yes,No,No,No,No,No,1 to 2,1 to 2,0 to 5,50%+
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,Yes,No,No,No,No,No,1 to 2,1 to 2,0 to 5,50%+


## 3.3 Feature Encoding
One-hot encoding `education_level` for downstream stakeholder consumption.

In [72]:
# One hot encode `education_level`
ed_level_dummies = internship_positions["education_level"].str.lower().str.get_dummies(sep=", ")
ed_level_dummies = ed_level_dummies.replace({0: "No", 1: "Yes"}).add_prefix("allows_")
ed_level_dummies = ed_level_dummies.add_suffix("_level")

internship_positions = pd.concat([internship_positions, ed_level_dummies], axis=1)
display(internship_positions.sample(5))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_social_and_law_majors,allows_education_majors,allows_tourism_and_hospitality_majors,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,allows_bachelor_level,allows_diploma_level,allows_profession_level
4456,a2433c5e-b69a-4706-857d-543afd918db8,2026-07-16T09:55:11+07:00,PPIC Internship,PT Kasakata Kimia,Kab. Bogor,"Bachelor, Diploma",Teknik Industri,"Melakukan proses koordinasi sales, produksi da...",5,1,...,No,No,No,1 to 2,1 to 2,0 to 5,11 - 25%,Yes,Yes,No
9537,a23f5a34-1019-490e-836b-0fa1e802be1b,2026-07-16T12:39:21+07:00,JURU BAHASA ISYARAT,RUDENIM MEDAN,Kota Medan,Bachelor,Pendidikan Luar Biasa,1. Menjadi penerjemah bagi pengguna layanan pe...,5,1,...,No,Yes,No,1 to 2,1 to 2,6 to 10,11 - 25%,Yes,No,No
18934,a240ebcd-805d-475a-ba3b-4a9a02ef1db0,2026-07-16T12:34:19+07:00,Pengelola Keuangan dan Anggaran,KANIM KELAS I TPI BANJARMASIN,Kota Banjarbaru,Bachelor,Akuntansi,1. Menyusun rencana kebutuhan anggaran tahunan...,5,1,...,No,No,No,1 to 2,1 to 2,6 to 10,0 - 10%,Yes,No,No
14218,a240d416-ef14-4de6-8908-582bbfdcabf6,2026-07-16T12:35:43+07:00,Asisten Publisitas dan Kehumasan,BPS Kabupaten Penukal Abab Lematang Ilir,Kab. Penukal Abab Lematang Ilir,"Diploma, Bachelor","Ilmu Komunikasi, Hubungan Masyarakat Dan Komun...",Membekali peserta dengan kemampuan menyusun ko...,5,2,...,No,No,No,1 to 2,1 to 2,11 to 20,11 - 25%,Yes,Yes,No
24772,a241478b-f160-4334-b85a-54650b421898,2026-07-16T12:03:16+07:00,PENGELOLA SDM,LEMBAGA PEMBINAAN KHUSUS ANAK KELAS II KENDARI,Kota Kendari,Bachelor,Manajemen,1. Mengumpulkan data dan informasi yang releva...,6,1,...,No,No,No,1 to 2,1 to 2,11 to 20,0 - 10%,Yes,No,No


## 3.4 Feature Extraction
Extracting a new binary flag, `allows_all_majors`, by parsing the `job_description` column.

In [73]:
all_majors_condition = internship_positions.job_description.str.contains(
    r"semua\sjurusan|jurusan\sapa.*|all\smajors|any\smajor",
    case=False
)

internship_positions["allows_all_majors"] = np.where(all_majors_condition, "Yes", "No")

display(internship_positions.sample(5))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_education_majors,allows_tourism_and_hospitality_majors,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,allows_bachelor_level,allows_diploma_level,allows_profession_level,allows_all_majors
14827,a24402d6-cb20-4ac9-81cc-313d711df38c,2026-07-16T12:04:07+07:00,Layanan Hukum,Kementerian Pendayagunaan Aparatur Negara dan ...,Kota Adm. Jakarta Selatan,Bachelor,Ilmu Hukum,Penugasan :\n1. Membantu penyusunan instrumen ...,5,2,...,No,No,1 to 2,1 to 2,11 to 20,11 - 25%,Yes,No,No,No
9264,a243a907-9457-44be-ac23-33773b6c86c0,2026-07-16T20:03:42+07:00,Supplier Management Specialist,Jaya Refrigeration Equipment,Kab. Bekasi,Bachelor,"Logistik, Manajemen","1. Supplier introduction and elimination, excl...",5,1,...,No,No,1 to 2,1 to 2,6 to 10,11 - 25%,Yes,No,No,No
5356,a24371bf-1c0a-4bd1-873d-d709378cbc79,2026-07-16T10:22:14+07:00,Pembantu Operator Pabrik Biji,PT Agro Sinergi Nusantara,Kab. Aceh Jaya,"Diploma, Bachelor","Teknik Mesin, Teknik Industri, Teknik Elektro,...",- Bertanggungjawab terhadap operasional Pabrik...,6,2,...,No,No,1 to 2,1 to 2,6 to 10,11 - 25%,Yes,Yes,No,No
13876,a2432007-a99d-4687-9d78-ef4ebcca92e5,2026-07-16T10:10:41+07:00,Tender Intern - DBSINT,Perusahaan Perseroan (Persero) PT Surveyor Ind...,Kota Adm. Jakarta Selatan,Bachelor,"Administrasi, Administrasi Bisnis, Manajemen P...",Mendukung unit tender dalam proses administras...,5,1,...,No,No,1 to 2,1 to 2,6 to 10,11 - 25%,Yes,No,No,No
5336,a23952cb-0633-4768-9ca3-2395be0f8eee,2026-07-16T10:30:24+07:00,Design Grafis,PT. Duta Graha Afiah,Kota Bogor,"Bachelor, Diploma","Multimedia, Sistem Informasi, Desain Komunikas...","1. Memahami alur kerja desain grafis, branding...",6,2,...,No,No,1 to 2,1 to 2,6 to 10,11 - 25%,Yes,Yes,No,No


# 4. Schema Finalization
Reorganizing the final 14 columns into a logical analytical structure before exporting.

In [74]:
# Get all the columns
internship_positions.columns

Index(['job_id', 'published_at', 'job_title', 'company', 'regency_city',
       'education_level', 'allowed_major', 'job_description',
       'weekly_working_day', 'requested_quota', 'approved_quota',
       'applicant_count', 'province', 'acceptance_percentage', 'job_category',
       'allows_it_and_computer_majors', 'allows_engineering_majors',
       'allows_business_majors', 'allows_health_majors',
       'allows_science_and_agri_majors', 'allows_arts_and_media_majors',
       'allows_social_and_law_majors', 'allows_education_majors',
       'allows_tourism_and_hospitality_majors', 'requested_quota_category',
       'approved_quota_category', 'applicant_count_category',
       'acceptance_percentage_category', 'allows_bachelor_level',
       'allows_diploma_level', 'allows_profession_level', 'allows_all_majors'],
      dtype='str')

In [80]:
# Reorganize the position of the columns
final_cols = [
    "job_id",
    "published_at",
    "job_title",
    "job_category",
    "company",
    "regency_city",
    "province",
    "allows_bachelor_level",
    "allows_diploma_level",
    "allows_profession_level",
    "allowed_major",
    "allows_it_and_computer_majors",
    "allows_engineering_majors",
    "allows_business_majors",
    "allows_health_majors",
    "allows_science_and_agri_majors",
    "allows_arts_and_media_majors",
    "allows_social_and_law_majors",
    "allows_education_majors",
    "allows_tourism_and_hospitality_majors",
    "allows_all_majors",
    "job_description",
    "weekly_working_day",
    "requested_quota_category",
    "approved_quota_category",
    "applicant_count_category",
    "acceptance_percentage_category",
    "requested_quota",
    "approved_quota",
    "applicant_count",
    "acceptance_percentage",
]

internship_postings = internship_positions[final_cols]

display(internship_postings.sample(10))

,job_id,published_at,job_title,job_category,company,regency_city,province,allows_bachelor_level,allows_diploma_level,allows_profession_level,...,job_description,weekly_working_day,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,requested_quota,approved_quota,applicant_count,acceptance_percentage
12059,a243e923-557d-4768-826d-7f17b2698e54,2026-07-16T12:32:14+07:00,Staff Administrasi Perencanaan,"HR, Admin & Management",Sekretariat Direktorat Jenderal Pengelolaan Ke...,Kota Adm. Jakarta Pusat,DKI Jakarta,Yes,No,No,...,melaksanakan pengelolaan dokumen di bidang pen...,5,1 to 2,1 to 2,11 to 20,11 - 25%,2,2,13,14.29
24579,a23543b0-9e37-4c7d-9c3b-47a7c4202521,2026-07-16T20:46:14+07:00,Finance & Accounting Assistant,Finance & Banking,PT Pandega Citraniaga,Kota Balikpapan,Kalimantan Timur,Yes,No,No,...,Membantu proses administrasi keuangan dan akun...,5,1 to 2,1 to 2,11 to 20,0 - 10%,1,1,16,5.88
25713,a23f142b-5484-4cd2-825d-4c87ac515db9,2026-07-16T12:01:15+07:00,PENGELOLA SDM,"HR, Admin & Management",LEMBAGA PEMBINAAN KHUSUS ANAK KELAS II BANDUNG,Kota Bandung,Jawa Barat,Yes,No,No,...,"""""""1. Mengumpulkan data dan informasi yang rel...",6,1 to 2,1 to 2,11 to 20,0 - 10%,1,1,18,5.26
3538,a23f516b-27e0-42ab-a235-3c5dc9ab395a,2026-07-16T12:52:10+07:00,PSIKOLOG,Healthcare & Medical,RUMAH TAHANAN NEGARA KELAS IIB TANJUNG BALAI K...,Kab. Karimun,Kepulauan Riau,Yes,No,No,...,1. Melakukan asesmen psikologis terhadap anak ...,5,1 to 2,1 to 2,0 to 5,11 - 25%,1,1,4,20.00
6230,a240d7e9-0168-4979-9b73-a10dfb8cff15,2026-07-16T12:14:42+07:00,PERAWAT,Healthcare & Medical,LEMBAGA PEMASYARAKATAN KELAS IIA LOMBOK BARAT,Kab. Lombok Barat,Nusa Tenggara Barat,Yes,No,No,...,1. Memberikan perawatan medis dasar bagi pegaw...,5,1 to 2,1 to 2,0 to 5,11 - 25%,1,1,5,16.67
17249,a2419618-7897-4d2f-a4b8-82deff120999,2026-07-16T12:36:48+07:00,Asisten Publisitas dan Kehumasan,"Media, PR & Creative",BPS Kota Kotamobagu,Kota Kotamobagu,Sulawesi Utara,Yes,Yes,No,...,Mendukung kegiatan kehumasan seperti penyusuna...,5,1 to 2,1 to 2,6 to 10,0 - 10%,1,1,9,10.00
21569,a2415822-5d14-4847-b2c6-e51074c8d61a,2026-07-16T12:34:59+07:00,Asisten Pengelola Keuangan,Finance & Banking,BPS Kabupaten Pacitan,Kab. Pacitan,Jawa Timur,Yes,Yes,No,...,Membantu proses pengelolaan administrasi keuan...,5,1 to 2,1 to 2,11 to 20,0 - 10%,1,1,12,7.69
5273,a242ce5a-98fb-46ea-8dcb-128378c85239,2026-07-16T11:08:33+07:00,Ahli Gizi,Healthcare & Medical,Rumah Sakit Islam Siti Rahmah,Kota Padang,Sumatera Barat,Yes,No,No,...,Peserta magang akan melaksanakan kegiatan pela...,6,1 to 2,1 to 2,6 to 10,11 - 25%,2,2,9,20.00
24211,a232f117-552c-40d3-8c3e-b813d152c519,2026-07-16T11:22:53+07:00,IT,Other,PT Sarana Patra Jateng,Kota Semarang,Jawa Tengah,Yes,Yes,No,...,Membangun dan mengembangkan website menggunaka...,5,1 to 2,1 to 2,11 to 20,0 - 10%,1,1,15,6.25
7238,a23f40b0-c9a2-4ddc-8f71-3d51c7b61906,2026-07-16T10:15:27+07:00,Product Specialist Lokerlink,Other,Sinergi Wahana Gemilang,Kota Adm. Jakarta Selatan,DKI Jakarta,Yes,No,No,...,Content Planning - Membuat content calendar bu...,5,1 to 2,1 to 2,0 to 5,11 - 25%,1,1,5,16.67


In [81]:
# Load the final, clean data to a local directory
internship_postings.to_parquet(
    INTERIM_DATA_DIR / "internship_postings.parquet", index=False
) 